# 03 – Preprocessing: Filtrado de Variables

**Proyecto:** Encuesta Permanente de Empleo Nacional (EPEN)  
**Objetivo:** Eliminar variables que no aportan valor predictivo: variables con alta cardinalidad, quasi-constantes, identificadores o con excesivos valores faltantes.

In [ ]:
import pandas as pd
import numpy as np
import os

# ─── Cargar datos ─────────────────────────────────────────────────────────────
PROC_PATH = os.path.join('..', 'data', 'processed', 'epen_missing_handled.csv')
try:
    df = pd.read_csv(PROC_PATH)
except FileNotFoundError:
    np.random.seed(42)
    n = 1000
    df = pd.DataFrame({
        'id_persona': range(n),                                    # Identificador (debe eliminarse)
        'edad': np.random.randint(14, 70, n),
        'sexo': np.random.choice(['Hombre', 'Mujer'], n),
        'nivel_educativo': np.random.choice(
            ['Sin instrucción', 'Primaria', 'Secundaria', 'Preparatoria', 'Universidad', 'Posgrado'], n
        ),
        'estado_civil': np.random.choice(['Soltero', 'Casado', 'Unión libre', 'Divorciado', 'Viudo'], n),
        'ingreso_mensual': np.random.exponential(8000, n).round(2),
        'horas_trabajadas': np.random.randint(0, 60, n),
        'tipo_empleo': np.random.choice(['Formal', 'Informal', 'Sin empleo'], n),
        'sector': np.random.choice(['Agricultura', 'Industria', 'Comercio', 'Servicios', 'Gobierno'], n),
        'constante': 1,                                           # Quasi-constante (debe eliminarse)
        'condicion_actividad': np.random.choice(
            ['Ocupado', 'Desocupado', 'No PEA'], n, p=[0.60, 0.10, 0.30]
        ),
        'target_desocupado': np.random.choice([0, 1], n, p=[0.90, 0.10]),
    })

print(f'Dataset cargado: {df.shape[0]:,} filas × {df.shape[1]} columnas')
print('Columnas:', df.columns.tolist())

## 1. Variables a eliminar manualmente

Estas variables se eliminan por criterio de dominio (no tienen poder predictivo):

In [ ]:
# Variables identificadoras o redundantes
drop_manual = [col for col in ['id_persona', 'condicion_actividad'] if col in df.columns]
print('Eliminando manualmente:', drop_manual)
df_filtered = df.drop(columns=drop_manual)

## 2. Variables quasi-constantes

In [ ]:
THRESHOLD_VARIANCE = 0.01
quasi_constant = []
for col in df_filtered.select_dtypes(include=np.number).columns:
    if col == 'target_desocupado':
        continue
    # Proporción del valor más frecuente
    top_freq = df_filtered[col].value_counts(normalize=True).iloc[0]
    if top_freq > (1 - THRESHOLD_VARIANCE):
        quasi_constant.append(col)
        print(f'  Quasi-constante: {col} (freq. top = {top_freq:.3f})')

df_filtered = df_filtered.drop(columns=quasi_constant)
print(f'\nVariables eliminadas por quasi-constancia: {quasi_constant}')

## 3. Variables con excesivos valores faltantes (>50%)

In [ ]:
THRESHOLD_MISSING = 0.50
high_missing = df_filtered.columns[df_filtered.isnull().mean() > THRESHOLD_MISSING].tolist()
if high_missing:
    print(f'Variables con >{THRESHOLD_MISSING*100:.0f}% de nulos: {high_missing}')
    df_filtered = df_filtered.drop(columns=high_missing)
else:
    print('No se encontraron variables con excesivos valores faltantes.')

In [ ]:
print(f'\nColumnas finales ({df_filtered.shape[1]}): {df_filtered.columns.tolist()}')

os.makedirs(os.path.join('..', 'data', 'processed'), exist_ok=True)
df_filtered.to_csv(os.path.join('..', 'data', 'processed', 'epen_filtered.csv'), index=False)
print('Dataset guardado: epen_filtered.csv')